# GreedyMini density

In [1]:
import struct
import os
import sys
import pandas as pd
import numpy as np
import hashlib
import matplotlib.pyplot as plt
from tqdm import tqdm
from joblib import Parallel, delayed
from pathlib import Path


In [2]:
# ----- Setup and utils -----
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def _physical_cpu_count():
    try:
        import psutil
        return psutil.cpu_count(logical=False) or os.cpu_count() or 1
    except Exception:
        return os.cpu_count() or 1

def _effective_n_jobs(n_jobs):
    if n_jobs == -1:
        return _physical_cpu_count()
    return n_jobs

def hash_kmer(k, kmer, seed):
    kmer_seed_sum = kmer + seed
    kmer_as_bytes = kmer_seed_sum.to_bytes(2 * k, byteorder='big')
    hashed_kmer = hashlib.sha256(kmer_as_bytes).digest()
    return int.from_bytes(hashed_kmer, byteorder='big')

def get_kmer_id_dna(k, kmer, seed):
    hashed_kmer = hash_kmer(k, kmer, seed) % (4 ** k)
    return (4 ** k) * hashed_kmer + kmer

def random_density_function_dna_includes_tiebreaking(w, k, function, n_samples=10000, seed=1):
    k_mask = (1 << (2 * k)) - 1
    total_nucleotides = n_samples + w + k - 1
    rng = np.random.default_rng(seed)
    nucleotides = rng.integers(0, 4, size=total_nucleotides, dtype=np.uint32)

    n_kmers = total_nucleotides - k + 1
    kmers = []
    kmer = 0
    for pos in range(k):
        kmer = (kmer << 2) | int(nucleotides[pos])
    kmers.append(kmer & k_mask)
    for i in range(1, n_kmers):
        kmer = ((kmer << 2) | int(nucleotides[i + k - 1])) & k_mask
        kmers.append(kmer)

    shift_8k = 8 * k
    ranks = [
        (function(km, w, k, seed) << shift_8k) + get_kmer_id_dna(k, km, seed)
        for km in kmers
    ]

    sampled_positions = 0
    last_sampled = -1
    for t in range(n_samples):
        window = ranks[t : t + w]
        min_pos = t + window.index(min(window))
        if min_pos != last_sampled:
            sampled_positions += 1
            last_sampled = min_pos

    return sampled_positions / n_kmers


In [3]:
# ----- GreedyMini order loader -----
def load_vector_from_file(filename):
    with open(filename, 'rb') as file:
        size_data = file.read(struct.calcsize('P'))  
        size = struct.unpack('P', size_data)[0]
        vector_data = file.read(size * struct.calcsize('Q')) 
        vector = struct.unpack(f'{size}Q', vector_data)
        return list(vector)


In [4]:
# ----- GreedyMini scheme -----
class GreedyMiniScheme:
    def __init__(self, k_train, w_train, file_path):
        self.k_train = k_train
        self.w_train = w_train
        self.name = f"greedymini_k{k_train}_w{w_train}"
        self.order_dict = load_vector_from_file(file_path)

    def get_minimizer(self, seed=None):
        def _fn(kmer, w, k, seed=None):
            # 1. binary->dna extension
            # Extract prefix of length k_train
            if k < self.k_train:
                # If the test k is smaller than the training k, pad it with zeros (A's)
                prefix_dna = kmer << (2 * (self.k_train - k))
            else:
                shift = 2 * (k - self.k_train)
                prefix_dna = (kmer >> shift) & ((1 << (2 * self.k_train)) - 1)
            
            bin_kmer = 0
            for i in range(self.k_train):
                nuc = (prefix_dna >> (2 * (self.k_train - 1 - i))) & 0b11
                # Projection for GreedyMini: A=C=0, G=T=1 (high bit of the 2-bit base)
                bit = (nuc >> 1) & 1
                bin_kmer = (bin_kmer << 1) | bit
                
            # 2. k-extension
            # We look up the rank of the prefix in the trained order
            rank_train = self.order_dict[bin_kmer]
            
            # 3. Tie-breaking using lexicographic order of the full kmer (implied by just returning it as the tiebreak)
            # Since rank_train is our primary key, we shift it by enough bits to dominate the tie-breaker.
            return (rank_train << (2 * k)) + kmer
        return _fn


In [5]:
def _run_one_density(w, k, minimizer_func, n_samples, seed):
    return random_density_function_dna_includes_tiebreaking(w, k, minimizer_func, n_samples, seed=seed)

def _compute_row_fixed_w(w, k, schemes, n_samples, repeats, n_jobs_inner):
    row = {"w": w, "k": k}
    for scheme in schemes:
        name = scheme.name if hasattr(scheme, "name") else scheme.__name__
        tasks = [delayed(_run_one_density)(w, k, scheme.get_minimizer(seed=i), n_samples, i) for i in range(repeats)]
        densities = Parallel(n_jobs=_effective_n_jobs(n_jobs_inner))(tasks)
        row[name] = np.mean(densities) * (w + 1)
    return row

def sample_density_fixed_w(w, schemes, k_range, n_samples=10000, repeats=5, n_jobs=-1):
    k_list = list(k_range)
    if n_jobs == 1:
        rows = [_compute_row_fixed_w(w, k, schemes, n_samples, repeats, n_jobs_inner=-1) for k in tqdm(k_list, desc=f"fixed w={w}", unit="k")]
    else:
        rows = Parallel(n_jobs=_effective_n_jobs(n_jobs))(delayed(_compute_row_fixed_w)(w, k, schemes, n_samples, repeats, n_jobs_inner=1) for k in tqdm(k_list, desc=f"fixed w={w}", unit="k"))
    return pd.DataFrame(rows)


In [6]:
# Dict of pairs of (k,w) for which we get the file, and range of k to test for (w always 24)
tasks = {
    (15, 17): range(15, 33),
    (8, 19): range(8, 33),
    (15, 30): range(15, 33),
}

for k_idx in range(2, 15):
    tasks[(k_idx, 15)] = range(k_idx, k_idx + 1)

W_TEST = 24
N_SAMPLES = 10_000_000
REPEATS = 1

schemes = []
for (k_train, w_train) in tasks.keys():
    file_path = f"GM/w{w_train}_k{k_train}.gm"
    if not os.path.exists(file_path):
        print(f"File {file_path} not found. Skipping.")
        continue
    schemes.append(GreedyMiniScheme(k_train, w_train, file_path))

# Determine overall k_range (union of all ranges)
min_k = min(min(r) for r in tasks.values())
max_k = max(max(r) for r in tasks.values())
overall_k_range = range(min_k, max_k + 1)

print(f"Testing over k in {list(overall_k_range)}")

# Compute density for all schemes simultaneously over the whole k range
df = sample_density_fixed_w(W_TEST, schemes, overall_k_range, n_samples=N_SAMPLES, repeats=REPEATS)

# Add rows for the lowest value and the name of the scheme that achieved it
scheme_cols = [col for col in df.columns if col not in ["w", "k"]]
df["lowest_value"] = df[scheme_cols].min(axis=1)
df["lowest_scheme"] = df[scheme_cols].idxmin(axis=1)

# Set rows to NaN if the k value is out of the requested range for that scheme
for (k_train, w_train), k_range in tasks.items():
    scheme_name = f"greedymini_k{k_train}_w{w_train}"
    if scheme_name in df.columns:
        mask = ~df["k"].isin(k_range)
        df.loc[mask, scheme_name] = np.nan

# Re-calculate lowest value and scheme ignoring NaNs
df["lowest_value"] = df[scheme_cols].min(axis=1)
df["lowest_scheme"] = df[scheme_cols].idxmin(axis=1)

out_file = f"{OUTPUT_DIR}/greedymini_density_all_wtest{W_TEST}.csv"
df.to_csv(out_file, index=False)
print(f"Saved {out_file}")


File GM/w15_k2.gm not found. Skipping.
Testing over k in [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]


fixed w=24: 100%|██████████| 31/31 [19:44<00:00, 38.21s/k]


Saved output/greedymini_density_all_wtest24.csv


C:\Users\tzion\AppData\Local\Temp\ipykernel_41912\859561157.py:47: FutureWarning: The behavior of DataFrame.idxmin with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  df["lowest_scheme"] = df[scheme_cols].idxmin(axis=1)


In [6]:
def _compute_row_fixed_k(w, k, schemes, n_samples, repeats, n_jobs_inner):
    row = {"w": w, "k": k}
    for scheme in schemes:
        name = scheme.name if hasattr(scheme, "name") else scheme.__name__
        tasks = [delayed(_run_one_density)(w, k, scheme.get_minimizer(seed=i), n_samples, i) for i in range(repeats)]
        densities = Parallel(n_jobs=_effective_n_jobs(n_jobs_inner))(tasks)
        row[name] = np.mean(densities) * (w + 1)
    return row

def sample_density_fixed_k(k, schemes, w_range, n_samples=10000, repeats=5, n_jobs=-1):
    w_list = list(w_range)
    if n_jobs == 1:
        rows = [_compute_row_fixed_k(w, k, schemes, n_samples, repeats, n_jobs_inner=-1) for w in tqdm(w_list, desc=f"fixed k={k}", unit="w")]
    else:
        rows = Parallel(n_jobs=_effective_n_jobs(n_jobs))(delayed(_compute_row_fixed_k)(w, k, schemes, n_samples, repeats, n_jobs_inner=1) for w in tqdm(w_list, desc=f"fixed k={k}", unit="w"))
    return pd.DataFrame(rows)


In [7]:
tasks_w = {
    (12, 20): range(10, 20),  # w_test in [10, 19]
    (12, 25): range(20, 25),  # w_test in [20, 24]
}

K_TEST = 12
N_SAMPLES_W = 10_000_000
REPEATS_W = 1

schemes_w = []
for (k_train, w_train) in tasks_w.keys():
    file_path = f"GM/w{w_train}_k{k_train}.gm"
    if not os.path.exists(file_path):
        print(f"File {file_path} not found. Skipping.")
        continue
    schemes_w.append(GreedyMiniScheme(k_train, w_train, file_path))

min_w = min(min(r) for r in tasks_w.values())
max_w = max(max(r) for r in tasks_w.values())
overall_w_range = range(min_w, max_w + 1)

print(f"Testing over w in {list(overall_w_range)}")

df_w = sample_density_fixed_k(K_TEST, schemes_w, overall_w_range, n_samples=N_SAMPLES_W, repeats=REPEATS_W)

scheme_cols_w = [col for col in df_w.columns if col not in ["w", "k"]]
for (k_train, w_train), w_range in tasks_w.items():
    scheme_name = f"greedymini_k{k_train}_w{w_train}"
    if scheme_name in df_w.columns:
        mask = ~df_w["w"].isin(w_range)
        df_w.loc[mask, scheme_name] = np.nan

df_w["lowest_value"] = df_w[scheme_cols_w].min(axis=1)
df_w["lowest_scheme"] = df_w[scheme_cols_w].idxmin(axis=1)

out_file_w = f"{OUTPUT_DIR}/greedymini_density_all_ktest{K_TEST}.csv"
df_w.to_csv(out_file_w, index=False)
print(f"Saved {out_file_w}")
df_w


Testing over w in [10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]


fixed k=12: 100%|██████████| 15/15 [00:00<00:00, 370.06w/s]


Saved output/greedymini_density_all_ktest12.csv


,w,k,greedymini_k12_w20,greedymini_k12_w25,lowest_value,lowest_scheme
0,10,12,1.595586,NaN,1.595586,greedymini_k12_w20
1,11,12,1.586312,NaN,1.586312,greedymini_k12_w20
2,12,12,1.584879,NaN,1.584879,greedymini_k12_w20
3,13,12,1.584986,NaN,1.584986,greedymini_k12_w20
4,14,12,1.583539,NaN,1.583539,greedymini_k12_w20
5,15,12,1.581780,NaN,1.581780,greedymini_k12_w20
6,16,12,1.582128,NaN,1.582128,greedymini_k12_w20
7,17,12,1.585155,NaN,1.585155,greedymini_k12_w20
8,18,12,1.591620,NaN,1.591620,greedymini_k12_w20
9,19,12,1.600071,NaN,1.600071,greedymini_k12_w20
